In [1]:
# Standard library imports
import os
import sys
import time

# # Add the parent directory to sys.path
# sys.path.append(os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt

import pandas as pd

from sarkas.processes import PreProcess, Simulation, PostProcess

# from src.init_methods_numba import uniform_perturb_bcc, random_reject, beta_perturb_bcc, normal_perturb_bcc, random_uniform_placement, sobol_placement, halton_placement
# Set the plotting style
plt.style.use('MSUstyle')

# Link to the input file.
input_file = os.path.join('input_files', 'lj_example.yaml')

In [2]:
# pre = PreProcess(input_file)
# pre.setup(read_yaml = True)
# pre.run(loops = 20)
# # pre.directory_sizes()

In [3]:
sim = Simulation(input_file)
sim.setup(read_yaml = True)
# sim.run()







________             ______                
__  ___/_____ __________  /_______ ________
_____ \_  __ `/_  ___/_  //_/  __ `/_  ___/
____/ // /_/ /_  /   _  ,<  / /_/ /_(__  ) 
/____/ \__,_/ /_/    /_/|_| \__,_/ /____/  
                                           


An open-source pure-python molecular dynamics suite for non-ideal plasmas.




********************************************************************************
                                   Simulation                                   
********************************************************************************

Job ID: lj
Job directory: SarkasSimulations/LJ_example
Simulation directory: 
SarkasSimulations/LJ_example/Simulation

Equilibration dumps directory: 
SarkasSimulations/LJ_example/Simulation/Equilibration/dumps
Production dumps directory: 
SarkasSimulations/LJ_example/Simulation/Production/dumps

Equilibration H5MD file: 
SarkasSimulations/LJ_example/Simulation/Equilibration/dumps/lj_data.h5md
Producti

In [ ]:
vsp_t = sim.particles.virial_species_tensor.sum()
print(f"Virial Species Tensor: {vsp_t:.6e}")

virial = sim.particles.virial_xx.sum() + sim.particles.virial_yy.sum() + sim.particles.virial_zz.sum() + 2.0 *( sim.particles.virial_xy.sum() + sim.particles.virial_xz.sum() + sim.particles.virial_yz.sum())
print(f"Virial: {virial:.6e}")

diff = vsp_t - virial
print(f"Difference: {diff:.6e}")

In [ ]:
sim.parameters.observables_arrays_list.append("virial")
sim.io.copy_params(sim.parameters)
# new_list = ["rdf_hist", "virial", "virial_xx", "virial_xy", "virial_xz", "virial_yy", "virial_yz", "virial_zz"]
# sim.particles.__getattribute__("virial")
sim.io.observables_arrays_list


In [ ]:
# # Remove the virial from the list
# if "virial" in new_list:
#     new_list.remove("virial")
# print(f"Updated observables_arrays_list: {new_list}")
sim.io.observables_arrays_list = [obs for obs in sim.parameters.observables_arrays_list]

sim.io.observables_arrays_list

In [ ]:
sim.io.setup_checkpoint(sim.parameters, sim.particles, phase = "equilibration")
# sim.io.open_h5md_file(phase = "production", mode = "a")
# sim.io.init_observables_group(sim.parameters, sim.particles, phase = "production", observables = ["virial"])
# sim.io.close_h5md_file()

In [ ]:
# Handle the case where observables contain virial            
observables = [obs for obs in sim.io.observables_arrays_list]
if "virial" in observables or "species_virial" in observables:
    for obs in observables:
        if "virial" in obs:
            # If the observable is a virial, we need to add the six components of the virial tensor
            for coord in ['xx', 'xy', 'xz', 'yy', 'yz', 'zz']:
                obs_name = f"{obs}_{coord}"
                if obs_name not in sim.io.observables_arrays_list:
                    sim.io.observables_arrays_list.append(obs_name)
            sim.io.observables_arrays_list.remove(obs)  # Remove the original virial name if it exists
        else:
            if obs not in sim.io.observables_arrays_list:
                sim.io.observables_arrays_list.append(obs)

# observables = sim.io.observables_arrays_list

print("Observables to track: ", sim.io.observables_arrays_list)

In [4]:
import h5py 

with h5py.File(sim.io.h5md_filepath, 'r') as f:
    # Check if the group exists
    if 'observables' in f:
        # Access the group
        observables_group = f['observables']
        # Print the keys in the group
        print("Keys in 'observables' group:", list(observables_group.keys()))
    else:
        print("'observables' group does not exist in the file.")

Keys in 'observables' group: ['Argon', 'rdf_hist', 'virial_xx', 'virial_xy', 'virial_xz', 'virial_yy', 'virial_yz', 'virial_zz']


In [ ]:
# sim.io.open_h5md_file(phase = "production", mode = "r")
# sim.io.close_h5md_file()
# bool(sim.io.h5md_file.id.valid)
sim.io.h5md_file.mode

In [ ]:
from scipy.spatial import distance_matrix

# np.where(pdist(pre.particles.pos) == 0)
dist = distance_matrix(sim.particles.pos, sim.particles.pos)
iu1 = np.triu_indices(dist.shape[0])
dist[iu1] = -1


In [ ]:
np.where(sim.particles.pos/ sim.species[0].sigma < 1.0)

In [ ]:
# sim.particles.remove_drift()

from tqdm import trange


sim.io.open_h5md_file(phase="equilibration")
it_start = sim.check_restart(phase="equilibration")
sim.integrator.update = sim.integrator.type_setup(sim.integrator.equilibration_type)
# Start timer, equilibrate, and print run time.
sim.timer.start()
# sim.evolve(
#     "equilibration",
#     sim.integrator.thermalization,
#     it_start,
#     sim.parameters.equilibration_steps,
#     sim.parameters.eq_dump_step,
# )
it_end = 15 # sim.parameters.equilibration_steps
it_start = 11
dump_step = 1
thermalization = sim.integrator.thermalization

for it in trange(it_start, it_end, disable=not sim.parameters.verbose):
    # Calculate the Potential energy and update particles' data

    sim.integrator.update(sim.particles)
    if (it + 1) % dump_step == 0:
        sim.particles.calculate_observables()
        # sim.io.dump(phase, sim.particles, it + 1)
        time = sim.integrator.dt * (it + 1)
        sim.io.save_timestep_data(it + 1, dump_step, time, sim.particles)

    if thermalization and (it + 1 >= sim.integrator.thermalization_timestep):
        sim.particles.calculate_species_kinetic_temperature()
        sim.integrator.thermostate(sim.particles)

time_eq = sim.timer.stop()
sim.io.close_h5md_file()
sim.io.time_stamp("Equilibration", sim.timer.time_division(time_eq))


In [ ]:
sum(sim.particles.pos/sim.particles.box_lengths > 1.0)
# sim.particles.pos

In [ ]:
sim.equilibrate()

In [ ]:
# Read the particles from the hdf5 file
sim.particles.restart_step = 0
sim.particles.load_from_checkpoint("equilibration", it = 187)
sim.particles.pos

In [ ]:
file_name = sim.particles.process_h5md_filepath_dict["equilibration"]

import h5py

with h5py.File(file_name, 'r') as f:
    # Read the positions of the particles
    positions = f['particles/pos'][15]
    # Read the velocities of the particles
    velocities = f['particles/vel'][15]
positions

In [ ]:
sum(positions/sim.particles.box_lengths > 1.0)
# # Calculate the memory usage of dist
# mem_usage = dist.nbytes / (1024 ** 2)  # Convert bytes to megabytes
# print(f"Memory usage of distance matrix: {mem_usage:.2f} MB")

In [ ]:
loops = 21
# pre.integrator.update = pre.integrator.type_setup(pre.integrator.equilibration_type)
# pre.io.open_h5md_file(phase="equilibration")
# pre.timer.start()
# pre.evolve("equilibration", pre.integrator.thermalization, 0, loops, pre.parameters.eq_dump_step)
# pre.io.close_h5md_file()
# # Print the average equilibration & production times
# pre.eq_mean_time = pre.timer.stop() / loops
# pre.io.preprocess_timing("Equilibration", pre.timer.time_division(pre.eq_mean_time), loops)

# pre.timer.stop()
pre.integrator.update = pre.integrator.type_setup(pre.integrator.production_type)
pre.potential_measure = True
pre.io.open_h5md_file(phase = "production")
pre.timer.start()
pre.evolve("production", False, 0, 21, pre.parameters.prod_dump_step)
pre.io.close_h5md_file()
pre.prod_mean_time = pre.timer.stop() / 21
pre.io.preprocess_timing("Production", pre.timer.time_division(pre.prod_mean_time), loops)
# pre.time_evolution_loop(loops = 21)


In [ ]:
prod_prediction = pre.prod_mean_time * pre.parameters.production_steps
pre.io.time_stamp("Production", pre.timer.time_division(prod_prediction))
